# LightGBM Electricity Consumption Forecast
Hourly per-station forecasting with **lagged features** and **Optuna** hyperparameter optimisation.

Forecasting window: **48 h** — val / test spans ~1 month each, so we roll the window recursively.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import torch
from sklearn.metrics import mean_squared_error
from tqdm.auto import tqdm
import gc

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── GPU config ──────────────────────────────────────────────────────────────
# LightGBM supports two GPU backends:
#   'cuda'  – CUDA backend (fastest, requires LightGBM built with CUDA support)
#   'gpu'   – OpenCL backend (fallback if CUDA build is unavailable)
#   'cpu'   – CPU only
if torch.cuda.is_available():
    try:
        # Quick smoke-test: train a tiny model with device='cuda'
        _probe = lgb.LGBMRegressor(n_estimators=1, device='cuda')
        _probe.fit([[0], [1]], [0, 1])
        DEVICE = 'cuda'
    except Exception:
        DEVICE = 'gpu'   # fall back to OpenCL build
else:
    DEVICE = 'cpu'

# With GPU training LightGBM manages its own parallelism;
# n_jobs > 1 on GPU gives no benefit and can cause contention.
N_JOBS = 1 if DEVICE != 'cpu' else -1

print(f"LightGBM  : {lgb.__version__}")
print(f"Optuna    : {optuna.__version__}")
print(f"CUDA avail: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU       : {torch.cuda.get_device_name(0)}")
print(f"LightGBM device → '{DEVICE}'")

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Warning] Using sparse features with CUDA is currently not supported.
LightGBM  : 4.6.0
Optuna    : 4.7.0
CUDA avail: True
GPU       : NVIDIA GeForce RTX 3090
LightGBM device → 'gpu'


## Utility functions (memory, metrics)

In [2]:
def smallest_int_dtype(min_val: int, max_val: int, signed: bool = True) -> str:
    if signed:
        for t in ["int8", "int16", "int32"]:
            info = np.iinfo(t)
            if info.min <= min_val <= max_val <= info.max:
                return t
        return "int64"
    else:
        for t in ["uint8", "uint16", "uint32"]:
            if 0 <= min_val <= max_val <= np.iinfo(t).max:
                return t
        return "uint64"


def optimize_df_for_memory(df: pd.DataFrame):
    meta = {}
    for col in df.columns:
        s = df[col]
        unique_non_null = set(df[col].dropna().unique())
        if unique_non_null.issubset({0, 1, True, False}) and col == "Група":
            df[col] = s.astype("bool")
            meta[col] = {"stored_as": "bool", "scale": 1}
            continue
        if pd.api.types.is_integer_dtype(s):
            mn, mx = int(s.min()), int(s.max())
            dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))
            df[col] = s.astype(dtype)
            meta[col] = {"stored_as": dtype, "scale": 1}
            continue
        if pd.api.types.is_float_dtype(s):
            non_null = s.dropna()
            if len(non_null) == 0:
                df[col] = s.astype("float32")
                meta[col] = {"stored_as": "float32", "scale": 1}
                continue
            decimals = non_null.astype(str).apply(
                lambda x: len(x.split(".")[1].rstrip("0")) if "." in x else 0
            ).max()
            if decimals <= 3:
                scale = 10 ** decimals
                scaled = np.round(s * scale)
                mn, mx = int(np.nanmin(scaled)), int(np.nanmax(scaled))
                int_dtype = smallest_int_dtype(mn, mx, signed=(mn < 0))
                if np.dtype(int_dtype).itemsize < np.dtype("float32").itemsize:
                    df[col] = scaled.astype(int_dtype)
                    meta[col] = {"stored_as": int_dtype, "scale": scale}
                    continue
            df[col] = s.astype("float32")
            meta[col] = {"stored_as": "float32", "scale": 1}
    return df, meta

In [3]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom  = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / np.maximum(denom, 1e-8))

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

## Paths & column config

In [4]:
TRAIN_PATH = "../../../data/silver_money_calc/train.parquet"
VAL_PATH   = "../../../data/silver_money_calc/val.parquet"
TEST_PATH  = "../../../data/silver_money_calc/test.parquet"

Y_COL     = "Sum of кВт"
GROUP_COL = "EIC-код_cat"

# Forecasting horizon used for recursive inference
HORIZON = 48  # hours

FUTURE_REALS = [
    "temperature_2m", "apparent_temperature",
    "dew_point_2m", "relative_humidity_2m", "precipitation", "rain",
    "snowfall", "cloud_cover", "cloud_cover_low", "cloud_cover_mid",
    "cloud_cover_high", "surface_pressure", "wind_speed_10m",
    "wind_direction_10m", "wind_gusts_10m", "shortwave_radiation",
    "diffuse_radiation", "direct_normal_irradiance",
]

weather_cols_to_drop = [
    "apparent_temperature", "rain", "snowfall",
    "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "surface_pressure", "wind_direction_10m", "wind_gusts_10m",
    "diffuse_radiation", "direct_normal_irradiance",
]

STATIC_REALS = ["Широта", "Довгота"]
TIME_VARYING_KNOWN_CATS = ["Month_cat", "Day_cat", "Hour_cat", "day_of_week_cat", "season_cat"]
STATIC_CATS = [
    "EIC-код_cat", "Група_cat", "АЗС_cat", "Тип_cat",
    "Область_cat", "ОСР код_cat", "ОСР опис_cat",
]

## Data loading & preprocessing

In [5]:
GLOBAL_MIN_DT = None

def build_time_index(df: pd.DataFrame) -> pd.DataFrame:
    global GLOBAL_MIN_DT
    df["datetime"] = pd.to_datetime(df["datetime"])
    if GLOBAL_MIN_DT is None:
        GLOBAL_MIN_DT = df["datetime"].min()
    df["time_idx"] = (
        (df["datetime"] - GLOBAL_MIN_DT).dt.total_seconds() / 3600
    ).astype(int)
    return df


def add_cat_helpers(df: pd.DataFrame) -> pd.DataFrame:
    df["datetime"]        = pd.to_datetime(df["datetime"])
    df["Month_cat"]       = df["datetime"].dt.month.astype(str)
    df["Day_cat"]         = df["datetime"].dt.day.astype(str)
    df["Hour_cat"]        = df["datetime"].dt.hour.astype(str)
    df["day_of_week_cat"] = df["datetime"].dt.dayofweek.astype(str)
    season_map = {
        12: "winter", 1: "winter", 2: "winter",
        3: "spring",  4: "spring", 5: "spring",
        6: "summer",  7: "summer", 8: "summer",
        9: "autumn", 10: "autumn", 11: "autumn",
    }
    df["season_cat"] = df["datetime"].dt.month.map(season_map)
    return df


def load_and_prepare(path: str, has_y: bool = True) -> pd.DataFrame:
    df = pd.read_parquet(path).reset_index(drop=True)
    df = df[df["datetime"] >= "2024-06-30"].reset_index(drop=True)
    df.columns = df.columns.str.replace(".", "_", regex=False)
    df, _ = optimize_df_for_memory(df)
    for col in df.select_dtypes(
        include=["int8", "int16", "int32", "uint8", "uint16", "uint32"]
    ).columns:
        df[col] = df[col].astype("float32")
    df = build_time_index(df)
    df = add_cat_helpers(df)
    for col in df.columns:
        if col.endswith("_cat"):
            df[col] = df[col].astype(str)
    if has_y:
        df[Y_COL] = df[Y_COL].astype("float32")
    else:
        df[Y_COL] = 0.0
    df = df.sort_values([GROUP_COL, "time_idx"]).reset_index(drop=True)
    try:
        df.drop(columns=["Ціна розподілу ЕЕ", "Ціна ЕЕ", "Money_spent"], inplace=True)
    except KeyError:
        pass
    df.drop(columns=[c for c in weather_cols_to_drop if c in df.columns], inplace=True)
    return df

In [6]:
print("Loading train …")
train = load_and_prepare(TRAIN_PATH)
print(f"train: {train.shape}  range: {train['datetime'].min()} → {train['datetime'].max()}")

print("Loading val   …")
val = load_and_prepare(VAL_PATH)

print("Loading test  …")
test = load_and_prepare(TEST_PATH)

print(f"val : {val.shape}")
print(f"test: {test.shape}")

Loading train …
train: (3300854, 23)  range: 2024-06-30 00:00:00+03:00 → 2025-06-30 00:00:00+03:00
Loading val   …
Loading test  …
val : (295368, 23)
test: (304152, 23)


In [7]:
station_stats = (
    train.groupby(GROUP_COL)
    .agg(rows=(Y_COL, "count"))
    .reset_index()
    .sort_values("rows", ascending=False)
)

TARGET_STATIONS = 409
np.random.seed(42)
sampled_stations = station_stats.sample(n=TARGET_STATIONS, random_state=42)[GROUP_COL].values
print(f"Stations: {len(sampled_stations)}")

train = train[train[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
val   = val[val[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
test  = test[test[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)

print(f"Train rows : {len(train):,}")
print(f"Val rows   : {len(val):,}")
print(f"Test rows  : {len(test):,}")

Stations: 409
Train rows : 3,300,854
Val rows   : 295,368
Test rows  : 303,408


In [8]:
# Merge into a single frame and rebuild time_idx globally
train["data_subset"] = "train"
val["data_subset"]   = "val"
test["data_subset"]  = "test"

all_data = (
    pd.concat([train, val, test], ignore_index=True)
    .sort_values([GROUP_COL, "time_idx"])
    .reset_index(drop=True)
)

all_data["datetime"] = pd.to_datetime(all_data["datetime"], utc=True, errors="coerce")
all_data["datetime"] = all_data["datetime"].dt.tz_convert(None)
all_data["time_idx"] = (
    (all_data["datetime"] - all_data["datetime"].min()).dt.total_seconds() // 3600
).astype("int64")

training_cutoff = all_data.loc[all_data["data_subset"] == "train", "time_idx"].max()
val_cutoff      = all_data.loc[all_data["data_subset"] == "val",   "time_idx"].max()
test_cutoff     = all_data.loc[all_data["data_subset"] == "test",  "time_idx"].max()

print(f"training cutoff : {training_cutoff}")
print(f"val cutoff      : {val_cutoff}")
print(f"test cutoff     : {test_cutoff}")

training cutoff : 8760
val cutoff      : 9504
test cutoff     : 10272


## Feature engineering — lagged & rolling variables

All lags are ≥ **HORIZON** (48 h) so they are safe to use during recursive inference without leaking future targets.

| Group | Lags |
|---|---|
| Short-term autocorrelation | 48, 49, 50, 51, 52 |
| Daily seasonality | 24×1 … 24×7 (same hour, previous N days) |
| Weekly seasonality | 168 (same hour, one week ago) |
| Rolling stats (window centred at lag 48) | mean/std/min/max over 24 h & 168 h |

In [9]:
# Lags to create (all >= HORIZON to be safe for recursive prediction)
LAG_HOURS = [
    HORIZON,
    HORIZON + 1,
    HORIZON + 2,
    HORIZON + 3,
    HORIZON + 4,
    24 * 2,   # 48 h  (same as HORIZON)
    24 * 3,
    24 * 4,
    24 * 5,
    24 * 6,
    24 * 7,   # 168 h  – same hour, one week ago
    24 * 14,  # two weeks
]
LAG_HOURS = sorted(set(LAG_HOURS))

# Rolling windows applied at offset=HORIZON (shift first, then roll)
ROLL_WINDOWS = [24, 48, 168]


def add_lag_features(df: pd.DataFrame, group_col: str, target_col: str) -> pd.DataFrame:
    """
    Add lagged target values and rolling statistics, grouped by station.
    Operates in-place and returns the modified DataFrame.
    """
    df = df.sort_values([group_col, "time_idx"]).reset_index(drop=True)
    grouped = df.groupby(group_col)[target_col]

    # --- lag features ---
    for lag in tqdm(LAG_HOURS, desc="Creating lag features"):
        df[f"lag_{lag}h"] = grouped.shift(lag).astype("float32")

    # --- rolling features (shifted by HORIZON to avoid leakage) ---
    shifted = grouped.shift(HORIZON)
    for w in tqdm(ROLL_WINDOWS, desc="Creating rolling features"):
        rolled = shifted.groupby(df[group_col]).transform(
            lambda x: x.rolling(w, min_periods=max(1, w // 4))
        )
        # pandas groupby-transform trick: rebuild rolling on the shifted series
        rolled_series = (
            df.assign(_shifted=shifted)
            .groupby(group_col)["_shifted"]
            .transform(lambda x: x.rolling(w, min_periods=max(1, w // 4)).mean())
        )
        df[f"roll_mean_{w}h"] = rolled_series.astype("float32")

        rolled_std = (
            df.assign(_shifted=shifted)
            .groupby(group_col)["_shifted"]
            .transform(lambda x: x.rolling(w, min_periods=max(1, w // 4)).std())
        )
        df[f"roll_std_{w}h"] = rolled_std.astype("float32")

    # --- diff features ---
    df["diff_24h"]  = (grouped.shift(HORIZON) - grouped.shift(HORIZON + 24)).astype("float32")
    df["diff_168h"] = (grouped.shift(HORIZON) - grouped.shift(HORIZON + 168)).astype("float32")

    return df


print("Adding lag features to all_data …")
all_data = add_lag_features(all_data, GROUP_COL, Y_COL)
print(f"Shape after feature engineering: {all_data.shape}")

Adding lag features to all_data …


Creating lag features:   0%|          | 0/11 [00:00<?, ?it/s]

Creating rolling features:   0%|          | 0/3 [00:00<?, ?it/s]

Shape after feature engineering: (3899630, 43)


## Identify feature columns and encode categoricals

In [10]:
# Numeric time features (LGBM handles these natively)
all_data["hour"]        = all_data["datetime"].dt.hour
all_data["dayofweek"]   = all_data["datetime"].dt.dayofweek
all_data["month"]       = all_data["datetime"].dt.month
all_data["day"]         = all_data["datetime"].dt.day
all_data["is_weekend"]  = (all_data["dayofweek"] >= 5).astype("int8")

# Cyclical encoding of hour, dayofweek, month
for col, period in [("hour", 24), ("dayofweek", 7), ("month", 12)]:
    all_data[f"{col}_sin"] = np.sin(2 * np.pi * all_data[col] / period).astype("float32")
    all_data[f"{col}_cos"] = np.cos(2 * np.pi * all_data[col] / period).astype("float32")

# Encode static categoricals as integer codes for LGBM
CAT_FEATURE_NAMES = []
for col in STATIC_CATS:
    if col in all_data.columns:
        all_data[col] = all_data[col].astype("category")
        CAT_FEATURE_NAMES.append(col)

# Remaining weather / numeric columns
WEATHER_COLS = [c for c in all_data.columns
                if c in FUTURE_REALS and c not in weather_cols_to_drop]

LAG_FEATURE_COLS = [c for c in all_data.columns
                    if c.startswith("lag_") or c.startswith("roll_") or c.startswith("diff_")]

TIME_FEATURE_COLS = [
    "hour", "dayofweek", "month", "day", "is_weekend",
    "hour_sin", "hour_cos", "dayofweek_sin", "dayofweek_cos",
    "month_sin", "month_cos",
]

FEATURE_COLS = TIME_FEATURE_COLS + STATIC_REALS + WEATHER_COLS + LAG_FEATURE_COLS + CAT_FEATURE_NAMES
# Keep only those that actually exist in the frame
FEATURE_COLS = [c for c in FEATURE_COLS if c in all_data.columns]

print(f"Total features : {len(FEATURE_COLS)}")
print(f"  Time         : {len(TIME_FEATURE_COLS)}")
print(f"  Weather      : {len(WEATHER_COLS)}")
print(f"  Lag/Rolling  : {len(LAG_FEATURE_COLS)}")
print(f"  Categorical  : {len(CAT_FEATURE_NAMES)}")

Total features : 45
  Time         : 11
  Weather      : 6
  Lag/Rolling  : 19
  Categorical  : 7


## Train / val split

In [11]:
train_df = all_data[all_data["time_idx"] <= training_cutoff].copy()
val_df   = all_data[(all_data["time_idx"] > training_cutoff) &
                    (all_data["time_idx"] <= val_cutoff)].copy()
test_df  = all_data[all_data["time_idx"] > val_cutoff].copy()

# Drop rows where lag features are NaN (first HORIZON rows per station)
lag_cols_required = [f"lag_{HORIZON}h"]  # minimum required lag
train_df = train_df.dropna(subset=lag_cols_required).reset_index(drop=True)

X_train = train_df[FEATURE_COLS]
y_train = train_df[Y_COL]

X_val   = val_df[FEATURE_COLS]
y_val   = val_df[Y_COL]

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {test_df[FEATURE_COLS].shape}")

X_train : (3281245, 45)
X_val   : (295368, 45)
X_test  : (303408, 45)


## Optuna hyperparameter optimisation

We optimise on **RMSE** of the first val window to keep tuning fast, then re-train with the best params on the full training set.

In [20]:
N_OPTUNA_TRIALS = 50   # increase for better results (e.g. 100–200)
OPTUNA_TIMEOUT  = 60 * 30  # 30 min wall-clock limit as a safety net

# Use a small, representative slice of val for fast evaluation during tuning
# (first 48-h window across all stations)
first_val_tidx = val_df["time_idx"].min()
val_tune_mask  = val_df["time_idx"] < first_val_tidx + HORIZON
X_val_tune     = val_df.loc[val_tune_mask, FEATURE_COLS]
y_val_tune     = val_df.loc[val_tune_mask, Y_COL]

from sklearn.preprocessing import OneHotEncoder
import scipy.sparse

ohe = OneHotEncoder(sparse_output=True, handle_unknown="ignore", dtype=np.float32)

cat_cols_present = [c for c in CAT_FEATURE_NAMES if c in X_train.columns]
non_cat_cols = [c for c in FEATURE_COLS if c not in cat_cols_present]

# Fit on train, transform all splits
ohe.fit(X_train[cat_cols_present].astype(str))

def apply_ohe(X):
    dense = X[non_cat_cols].astype("float32")
    encoded = ohe.transform(X[cat_cols_present].astype(str))
    return scipy.sparse.hstack([dense.values, encoded], format="csr")

X_train    = apply_ohe(X_train)
X_val_tune = apply_ohe(X_val_tune)
X_val      = apply_ohe(X_val)
X_test     = apply_ohe(test_df[FEATURE_COLS])

def objective(trial: optuna.Trial) -> float:
    params = {
        "objective":        "regression",
        "metric":           "rmse",
        "verbosity":        -1,
        "boosting_type":    "gbdt",
        "device":           DEVICE,
        "max_bin":          255,
        "n_estimators":     trial.suggest_int("n_estimators", 300, 2000),
        "learning_rate":    trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "num_leaves":       trial.suggest_int("num_leaves", 16, 512),
        "max_depth":        trial.suggest_int("max_depth", 3, 12),
        "min_child_samples":trial.suggest_int("min_child_samples", 5, 200),
        "subsample":        trial.suggest_float("subsample", 0.4, 1.0),
        "subsample_freq":   trial.suggest_int("subsample_freq", 1, 10),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_split_gain":   trial.suggest_float("min_split_gain", 0.0, 1.0),
        "n_jobs":           N_JOBS,
        "random_state":     42,
    }

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val_tune, y_val_tune)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
        # categorical_feature removed
    )
    preds = model.predict(X_val_tune)
    return rmse(y_val_tune, preds)


print(f"Running Optuna ({N_OPTUNA_TRIALS} trials, timeout={OPTUNA_TIMEOUT//60} min) …")
study = optuna.create_study(direction="minimize", study_name="lgbm_electricity")
study.optimize(
    objective,
    n_trials=N_OPTUNA_TRIALS,
    timeout=OPTUNA_TIMEOUT,
    show_progress_bar=True,
)

print(f"\nBest RMSE : {study.best_value:.4f}")
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k:25s} = {v}")

Running Optuna (50 trials, timeout=30 min) …


  0%|          | 0/50 [00:00<?, ?it/s]


Best RMSE : 4.9299
Best params:
  n_estimators              = 754
  learning_rate             = 0.0841036591258443
  num_leaves                = 202
  max_depth                 = 6
  min_child_samples         = 5
  subsample                 = 0.7962183741513966
  subsample_freq            = 4
  colsample_bytree          = 0.583860722797084
  reg_alpha                 = 1.4685078407459413e-08
  reg_lambda                = 0.0030814374049976086
  min_split_gain            = 0.9679019703420079


## Final model training with best params

In [21]:
best_params = {
    "objective":     "regression",
    "metric":        "rmse",
    "verbosity":     -1,
    "boosting_type": "gbdt",
    "device":        DEVICE,
    "n_jobs":        N_JOBS,
    "random_state":  42,
    "max_bin":       255,
    **study.best_params,
}

print("Training final LightGBM model …")
final_model = lgb.LGBMRegressor(**best_params)
final_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=50),
    ],
    categorical_feature=CAT_FEATURE_NAMES,
)
print("Training complete.")

# Optuna importance plot
try:
    from optuna.visualization import plot_param_importances, plot_optimization_history
    plot_optimization_history(study).show()
    plot_param_importances(study).show()
except Exception:
    pass

Training final LightGBM model …


TypeError: Wrong type(str) or unknown name(EIC-код_cat) in categorical_feature

## Recursive inference (48-h rolling window)

Val and test each span ~1 month. Because our lags start at 48 h, once we have predicted a window we can use those predictions as lag values for the *next* window, rolling forward by `HORIZON` steps at a time.

In [ ]:
def recursive_predict(
    model: lgb.LGBMRegressor,
    history_df: pd.DataFrame,    # all rows with known targets up to the forecast origin
    future_df:  pd.DataFrame,    # rows to predict (exogenous features already filled)
    feature_cols: list,
    group_col: str,
    target_col: str,
    horizon: int,
) -> pd.DataFrame:
    """
    Predict `future_df` step-by-step in chunks of `horizon` rows per station.
    Lag features are recomputed from the growing prediction buffer after each step.

    Returns future_df with a new 'pred' column.
    """
    # Work with a mutable copy that we will update with predictions
    buffer = history_df.copy()
    future_out = future_df.copy()
    future_out["pred"] = np.nan

    future_tidx = sorted(future_df["time_idx"].unique())
    # Group time indices into windows of size `horizon`
    windows = [future_tidx[i:i + horizon] for i in range(0, len(future_tidx), horizon)]

    for window in tqdm(windows, desc="Recursive inference windows"):
        window_set = set(window)
        win_rows   = future_df[future_df["time_idx"].isin(window_set)].copy()

        # -------- rebuild lag features for this window --------
        # Combine buffer + window (target = NaN for window rows, will be filled later)
        combined = pd.concat(
            [buffer[[group_col, "time_idx", target_col]],
             win_rows[[group_col, "time_idx"]].assign(**{target_col: np.nan})],
            ignore_index=True,
        ).sort_values([group_col, "time_idx"]).reset_index(drop=True)

        # Recompute lags on the combined frame
        grp = combined.groupby(group_col)[target_col]
        for lag in LAG_HOURS:
            combined[f"lag_{lag}h"] = grp.shift(lag).astype("float32")

        shifted = grp.shift(horizon)
        for w in ROLL_WINDOWS:
            combined[f"roll_mean_{w}h"] = (
                combined.assign(_s=shifted)
                .groupby(group_col)["_s"]
                .transform(lambda x: x.rolling(w, min_periods=max(1, w // 4)).mean())
                .astype("float32")
            )
            combined[f"roll_std_{w}h"] = (
                combined.assign(_s=shifted)
                .groupby(group_col)["_s"]
                .transform(lambda x: x.rolling(w, min_periods=max(1, w // 4)).std())
                .astype("float32")
            )
        combined["diff_24h"]  = (grp.shift(horizon) - grp.shift(horizon + 24)).astype("float32")
        combined["diff_168h"] = (grp.shift(horizon) - grp.shift(horizon + 168)).astype("float32")

        # Extract lag cols for the window rows only
        lag_cols_all = [c for c in combined.columns
                        if c.startswith("lag_") or c.startswith("roll_") or c.startswith("diff_")]
        win_lag = combined[combined["time_idx"].isin(window_set)][[
            group_col, "time_idx"] + lag_cols_all].copy()

        # Merge lag features back into win_rows
        non_lag_feat = [c for c in feature_cols if c not in lag_cols_all]
        win_rows = win_rows[non_lag_feat + [group_col, "time_idx"]].merge(
            win_lag, on=[group_col, "time_idx"], how="left"
        )

        # Predict
        X_win  = win_rows[[c for c in feature_cols if c in win_rows.columns]]
        preds  = model.predict(X_win).clip(min=0)  # energy cannot be negative
        win_rows["pred"] = preds

        # Update future_out
        future_out.loc[
            future_out["time_idx"].isin(window_set), "pred"
        ] = win_rows.set_index([group_col, "time_idx"])["pred"].reindex(
            future_out.loc[future_out["time_idx"].isin(window_set)]
            .set_index([group_col, "time_idx"]).index
        ).values

        # Feed predictions back as pseudo-targets for next window's lags
        new_rows = win_rows[[group_col, "time_idx"]].copy()
        new_rows[target_col] = preds
        buffer = pd.concat(
            [buffer[[group_col, "time_idx", target_col]], new_rows],
            ignore_index=True,
        )

    return future_out

In [ ]:
print("Predicting validation set …")
val_result = recursive_predict(
    model       = final_model,
    history_df  = train_df[[GROUP_COL, "time_idx", Y_COL]],
    future_df   = val_df,
    feature_cols= FEATURE_COLS,
    group_col   = GROUP_COL,
    target_col  = Y_COL,
    horizon     = HORIZON,
)

val_eval = val_result.dropna(subset=["pred", Y_COL]).copy()
print(f"Val predictions complete — {len(val_eval):,} rows")

In [ ]:
# For test, history = train + val (with their true targets)
history_for_test = pd.concat(
    [train_df[[GROUP_COL, "time_idx", Y_COL]],
     val_df[[GROUP_COL, "time_idx", Y_COL]]],
    ignore_index=True,
)

print("Predicting test set …")
test_result = recursive_predict(
    model       = final_model,
    history_df  = history_for_test,
    future_df   = test_df,
    feature_cols= FEATURE_COLS,
    group_col   = GROUP_COL,
    target_col  = Y_COL,
    horizon     = HORIZON,
)

test_eval = test_result.dropna(subset=["pred", Y_COL]).copy()
print(f"Test predictions complete — {len(test_eval):,} rows")

## Evaluation metrics

In [ ]:
print("── Validation ──────────────────────────────────────────────")
print(f"Aligned samples : {len(val_eval):,}")
print(f"SMAPE : {smape(val_eval[Y_COL], val_eval['pred']):.4f}")
print(f"RMSE  : {rmse( val_eval[Y_COL], val_eval['pred']):.4f}")
print(f"MAPE  : {mape( val_eval[Y_COL], val_eval['pred']):.2f} %")

print("\n── Test ────────────────────────────────────────────────────")
print(f"Aligned samples : {len(test_eval):,}")
print(f"SMAPE : {smape(test_eval[Y_COL], test_eval['pred']):.4f}")
print(f"RMSE  : {rmse( test_eval[Y_COL], test_eval['pred']):.4f}")
print(f"MAPE  : {mape( test_eval[Y_COL], test_eval['pred']):.2f} %")

## Feature importance

In [ ]:
import matplotlib.pyplot as plt

fi = pd.Series(
    final_model.feature_importances_,
    index=FEATURE_COLS,
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
fi.head(30).sort_values().plot.barh(ax=ax, color="steelblue")
ax.set_title("LightGBM — Top-30 feature importances (gain)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

## Forecast visualisation

In [ ]:
import matplotlib.dates as mdates


def plot_forecast(eval_df, eic_code, start_dt=None, end_dt=None):
    df = eval_df.merge(
        all_data[[GROUP_COL, "time_idx", "datetime"]],
        on=[GROUP_COL, "time_idx"], how="inner",
    )
    df = df[df[GROUP_COL] == eic_code].sort_values("datetime")
    if df.empty:
        raise ValueError(f"No data for EIC code: {eic_code!r}")
    if start_dt is not None:
        df = df[df["datetime"] >= pd.Timestamp(start_dt)]
    if end_dt is not None:
        df = df[df["datetime"] <= pd.Timestamp(end_dt)]
    if df.empty:
        raise ValueError("No data in the specified datetime range.")

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df["datetime"], df[Y_COL],  label="True",      linewidth=1, color="steelblue")
    ax.plot(df["datetime"], df["pred"], label="Predicted", linewidth=1, color="tomato", alpha=0.85)
    ax.set_title(
        f"{eic_code}  |  MAPE={mape(df[Y_COL], df['pred']):.3f}"
        f"  RMSE={rmse(df[Y_COL], df['pred']):.2f}"
        f"  ({df['datetime'].min().date()} – {df['datetime'].max().date()})"
    )
    ax.set_xlabel("Datetime")
    ax.set_ylabel(Y_COL)
    ax.legend()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    fig.autofmt_xdate(rotation=0, ha="center")
    plt.tight_layout()
    plt.show()


# Example:
# plot_forecast(val_eval,  eic_code="<code>")
# plot_forecast(test_eval, eic_code="<code>", start_dt="2025-08-25", end_dt="2025-08-26")

In [ ]:
plot_forecast(test_eval, eic_code='62Z9042463861989')

In [ ]:
plot_forecast(test_eval, eic_code='62Z2410378897684')